









# Experiment 07 — Automated Recruitment Workflow System (HR Domain)



This notebook creates a synthetic dataset of 100 fictional candidates, validates and normalizes records, detects duplicate IDs, calculates explainable weighted scores, routes applications, creates notification drafts, simulates human review, writes audit/rejected-record files, runs tests, and reports evaluation metrics.

**Important:** This is a decision-support demonstration only. It does not send real emails or make final hiring/rejection decisions. All consequential outcomes require authorized human review.

## 1. Setup and imports

No extra package installation is required. The notebook uses Python's standard library and pandas for convenient result tables (already available in Colab).

In [1]:
import csv
import json
import random
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display

random.seed(7)
BASE_DIR = Path("/content/RecruitmentWorkflow")
BASE_DIR.mkdir(parents=True, exist_ok=True)

print("Workspace:", BASE_DIR)
print("Python environment ready.")

Workspace: /content/RecruitmentWorkflow
Python environment ready.


## 2. Create the project files and scoring configuration

The scoring criteria follow the lab sheet: skills 40%, relevant experience 30%, technical assessment 30%. Routing thresholds are 75 and 50. These are illustrative lab settings, not universal recruitment standards.

In [2]:
RULES = {
    "rule_version": "1.0",
    "criteria": {
        "skills_score": 40,
        "experience_score": 30,
        "assessment_score": 30
    },
    "thresholds": {
        "shortlist_review": 75,
        "manual_review": 50
    }
}
RULES_PATH = BASE_DIR / "rules.json"
RULES_PATH.write_text(json.dumps(RULES, indent=2), encoding="utf-8")

REQUIRED_FIELDS = [
    "candidate_id", "candidate_name", "email",
    "skills_score", "experience_score", "assessment_score",
    "application_role"
]
SCORE_FIELDS = ["skills_score", "experience_score", "assessment_score"]
ROLE = "Junior Software Developer"

print(RULES_PATH.read_text(encoding="utf-8"))

{
  "rule_version": "1.0",
  "criteria": {
    "skills_score": 40,
    "experience_score": 30,
    "assessment_score": 30
  },
  "thresholds": {
    "shortlist_review": 75,
    "manual_review": 50
  }
}


## 3. Generate 100 fictional candidate records

The dataset includes normal records and deliberate test cases: missing email, duplicate candidate ID, score outside 0–100, invalid numeric score, threshold-boundary scores, a high-scoring candidate, and a low-scoring candidate. All names and emails are fictional.

In [3]:
candidates = []
first_names = ["Aarav", "Diya", "Kabir", "Ira", "Neel", "Anaya", "Rohan", "Meera", "Vivaan", "Sara"]
last_names = ["Shah", "Mehta", "Patil", "Desai", "Joshi", "Rao", "Kapoor", "Nair", "Kulkarni", "Singh"]

for i in range(1, 101):
    candidates.append({
        "candidate_id": f"C{i:03d}",
        "candidate_name": f"{first_names[(i-1) % len(first_names)]} {last_names[(i*3) % len(last_names)]}",
        "email": f"candidate{i:03d}@example.com",
        "skills_score": random.randint(35, 98),
        "experience_score": random.randint(30, 95),
        "assessment_score": random.randint(30, 99),
        "application_role": ROLE
    })

# Explicit, reproducible edge cases
candidates[0].update({"candidate_id":"C101", "candidate_name":"Aarav Shah", "email":"aarav@example.com",
                      "skills_score":85, "experience_score":70, "assessment_score":90})  # score 82
candidates[1].update({"candidate_id":"C102", "candidate_name":"Diya Mehta", "email":"diya@example.com",
                      "skills_score":70, "experience_score":80, "assessment_score":75})  # score 74.5
candidates[2].update({"candidate_id":"C103", "candidate_name":"Kabir Patil", "email":"kabir@example.com",
                      "skills_score":55, "experience_score":50, "assessment_score":60})
candidates[3].update({"candidate_id":"C104", "candidate_name":"Ira Desai", "email":"ira@example.com",
                      "skills_score":100, "experience_score":100, "assessment_score":100})  # high
candidates[4].update({"candidate_id":"C105", "candidate_name":"Neel Joshi", "email":"neel@example.com",
                      "skills_score":0, "experience_score":0, "assessment_score":0})  # low

# Exactly 75, exactly 50, and 74.99 score cases
candidates[5].update({"skills_score":75, "experience_score":75, "assessment_score":75})
candidates[6].update({"skills_score":50, "experience_score":50, "assessment_score":50})
candidates[7].update({"skills_score":74.99, "experience_score":74.99, "assessment_score":74.99})

# Invalid examples
candidates[8]["email"] = "   "
candidates[9]["skills_score"] = 120
candidates[10]["experience_score"] = -5
candidates[11]["assessment_score"] = "not_a_number"
# Duplicate ID: preserve it and flag it during validation
candidates[12]["candidate_id"] = candidates[0]["candidate_id"]

CANDIDATES_PATH = BASE_DIR / "candidates.csv"
with CANDIDATES_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=REQUIRED_FIELDS)
    writer.writeheader()
    writer.writerows(candidates)

print(f"Created {len(candidates)} synthetic candidate records at {CANDIDATES_PATH}")
display(pd.DataFrame(candidates).head(10))

Created 100 synthetic candidate records at /content/RecruitmentWorkflow/candidates.csv


,candidate_id,candidate_name,email,skills_score,experience_score,assessment_score,application_role
0,C101,Aarav Shah,aarav@example.com,85.00,70.00,90,Junior Software Developer
1,C102,Diya Mehta,diya@example.com,70.00,80.00,75,Junior Software Developer
2,C103,Kabir Patil,kabir@example.com,55.00,50.00,60,Junior Software Developer
3,C104,Ira Desai,ira@example.com,100.00,100.00,100,Junior Software Developer
4,C105,Neel Joshi,neel@example.com,0.00,0.00,0,Junior Software Developer
5,C006,Anaya Kulkarni,candidate006@example.com,75.00,75.00,75,Junior Software Developer
6,C007,Rohan Mehta,candidate007@example.com,50.00,50.00,50,Junior Software Developer
7,C008,Meera Joshi,candidate008@example.com,74.99,74.99,74.99,Junior Software Developer
8,C009,Vivaan Nair,,63.00,35.00,47,Junior Software Developer
9,C010,Sara Shah,candidate010@example.com,120.00,83.00,48,Junior Software Developer


## 4. Load candidate records safely

The loader checks required headers and handles missing files or malformed CSV. Scoring must not proceed if ingestion fails.

In [4]:
def load_candidates(file_path):
    try:
        with open(file_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            headers = reader.fieldnames or []
            missing_headers = [field for field in REQUIRED_FIELDS if field not in headers]
            if missing_headers:
                raise ValueError(f"Missing required CSV headers: {missing_headers}")
            records = list(reader)
        print("Total records loaded:", len(records))
        return records
    except FileNotFoundError as exc:
        raise FileNotFoundError(f"Candidate file not found: {file_path}") from exc
    except csv.Error as exc:
        raise ValueError(f"Malformed CSV file: {exc}") from exc

loaded_candidates = load_candidates(CANDIDATES_PATH)

Total records loaded: 100


## 5. Normalize, validate, and detect duplicates

Fields are trimmed. Required fields, email format, numeric values, score ranges, and duplicate application IDs are checked. Invalid and duplicate records are quarantined with explicit reasons; they are not silently deleted.

In [5]:
def normalize_candidate(candidate):
    return {
        key: (value.strip() if isinstance(value, str) else value)
        for key, value in candidate.items()
    }

def validate_candidate(candidate, seen_ids):
    record = normalize_candidate(candidate)

    for field in REQUIRED_FIELDS:
        if not str(record.get(field, "")).strip():
            return record, False, f"Missing field: {field}"

    candidate_id = record["candidate_id"]
    if candidate_id in seen_ids:
        return record, False, f"Duplicate candidate_id: {candidate_id}"

    if not re.fullmatch(r"[^\s@]+@[^\s@]+\.[^\s@]+", record["email"]):
        return record, False, "Invalid email format"

    for field in SCORE_FIELDS:
        try:
            score = float(record[field])
        except (ValueError, TypeError):
            return record, False, f"Invalid numeric value: {field}"
        if not 0 <= score <= 100:
            return record, False, f"{field} must be between 0 and 100"
        record[field] = score

    seen_ids.add(candidate_id)
    return record, True, "Valid record"

seen_ids = set()
valid_records, rejected_records = [], []
validation_rows = []

for raw in loaded_candidates:
    normalized, is_valid, reason = validate_candidate(raw, seen_ids)
    validation_rows.append({
        "candidate_id": normalized.get("candidate_id", ""),
        "candidate_name": normalized.get("candidate_name", ""),
        "validation_status": "VALID" if is_valid else "INVALID",
        "validation_reason": reason
    })
    if is_valid:
        valid_records.append(normalized)
    else:
        rejected_records.append({**normalized, "rejection_reason": reason})

REJECTED_PATH = BASE_DIR / "rejected_records.csv"
rejected_columns = REQUIRED_FIELDS + ["rejection_reason"]
with REJECTED_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rejected_columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rejected_records)

validation_df = pd.DataFrame(validation_rows)
display(validation_df[validation_df["validation_status"] == "INVALID"])
print(f"Valid records: {len(valid_records)} | Rejected/quarantined records: {len(rejected_records)}")
print("Rejected records saved to:", REJECTED_PATH)

,candidate_id,candidate_name,validation_status,validation_reason
8,C009,Vivaan Nair,INVALID,Missing field: email
9,C010,Sara Shah,INVALID,skills_score must be between 0 and 100
10,C011,Aarav Desai,INVALID,experience_score must be between 0 and 100
11,C012,Diya Kapoor,INVALID,Invalid numeric value: assessment_score
12,C101,Kabir Singh,INVALID,Duplicate candidate_id: C101


Valid records: 95 | Rejected/quarantined records: 5
Rejected records saved to: /content/RecruitmentWorkflow/rejected_records.csv


## 6. Load and validate the scoring rules

The weights must total 100. Contributions are shown separately so a reviewer can understand the total score.

In [6]:
def load_rules(file_path):
    data = json.loads(Path(file_path).read_text(encoding="utf-8"))
    criteria = data.get("criteria", {})
    if set(criteria) != set(SCORE_FIELDS):
        raise ValueError("Scoring criteria must define all three score fields.")
    if sum(float(v) for v in criteria.values()) != 100:
        raise ValueError("Scoring weights must add up to 100.")
    thresholds = data.get("thresholds", {})
    if not (0 <= float(thresholds["manual_review"]) <= float(thresholds["shortlist_review"]) <= 100):
        raise ValueError("Thresholds must be ordered between 0 and 100.")
    return data

rules = load_rules(RULES_PATH)

def calculate_score(candidate, rules):
    weights = rules["criteria"]
    breakdown = {
        "skills_contribution": round(float(candidate["skills_score"]) * weights["skills_score"] / 100, 2),
        "experience_contribution": round(float(candidate["experience_score"]) * weights["experience_score"] / 100, 2),
        "assessment_contribution": round(float(candidate["assessment_score"]) * weights["assessment_score"] / 100, 2)
    }
    total = round(sum(breakdown.values()), 2)
    if not 0 <= total <= 100:
        raise ValueError("Computed score outside expected range.")
    return total, breakdown

def route_candidate(score, rules):
    if score >= float(rules["thresholds"]["shortlist_review"]):
        return "SHORTLIST_REVIEW"
    if score >= float(rules["thresholds"]["manual_review"]):
        return "MANUAL_REVIEW"
    return "NOT_SHORTLISTED_REVIEW"

# Validate the scoring example specified in the lab sheet
example = {"skills_score":85, "experience_score":70, "assessment_score":90}
example_score, example_breakdown = calculate_score(example, rules)
print("C101 example score:", example_score, "/ 100")
print("Breakdown:", example_breakdown)
assert example_score == 82.0

C101 example score: 82.0 / 100
Breakdown: {'skills_contribution': 34.0, 'experience_contribution': 21.0, 'assessment_contribution': 27.0}


## 7. Generate notification drafts

Drafts are status-specific and do not reveal scores. They are never sent automatically. Even the low-score category uses neutral language and does not imply that a final rejection has occurred.

In [7]:
def generate_notification(candidate, status):
    name = candidate["candidate_name"]
    candidate_id = candidate["candidate_id"]

    if status == "SHORTLIST_REVIEW":
        subject = "Application Update"
        message = (
            f"Dear {name},\n\n"
            f"Your application (reference {candidate_id}) is being considered for "
            "the next stage of the recruitment process. The HR team will review "
            "your application and communicate the next steps.\n\nRegards,\nRecruitment Team"
        )
    elif status == "MANUAL_REVIEW":
        subject = "Application Under Review"
        message = (
            f"Dear {name},\n\n"
            f"Your application (reference {candidate_id}) is currently under review. "
            "The recruitment team will communicate any further updates.\n\nRegards,\nRecruitment Team"
        )
    else:
        subject = "Application Review Update"
        message = (
            f"Dear {name},\n\n"
            f"Your application (reference {candidate_id}) is undergoing review by "
            "the recruitment team. You will be informed of the outcome after the review.\n\n"
            "Regards,\nRecruitment Team"
        )

    return {
        "candidate_id": candidate_id,
        "subject": subject,
        "message": message,
        "notification_status": "DRAFT"
    }

print("Notification draft generator ready.")

Notification draft generator ready.


## 8. Human-in-the-loop review and audit logging

Review actions: `APPROVE`, `EDIT`, `HOLD`, and `REQUEST_INFO`. Only an authorized HR reviewer can take an action. The notebook defaults to `HOLD` for every application so it does not assume that any recommendation has been approved. You can change `DEMO_REVIEW_ACTION` to demonstrate another action.

In [8]:
def review_action(proposed_status, action):
    action = action.upper()
    if action == "APPROVE":
        return proposed_status
    if action == "EDIT":
        return "REQUIRES_REVIEW"
    if action == "HOLD":
        return "ON_HOLD"
    if action == "REQUEST_INFO":
        return "AWAITING_INFORMATION"
    raise ValueError("Invalid review action")

AUTHORIZED_REVIEWER_ROLES = {"HR_REVIEWER", "HR_ADMIN"}

def apply_human_review(proposed_status, action, reviewer_id, reviewer_role, notes=""):
    if reviewer_role not in AUTHORIZED_REVIEWER_ROLES:
        raise PermissionError("Unauthorized approval attempt: reviewer role is not permitted.")
    return review_action(proposed_status, action)

AUDIT_PATH = BASE_DIR / "audit_log.csv"
AUDIT_FIELDS = [
    "timestamp", "candidate_id", "event", "validation_status", "score",
    "score_breakdown", "rule_version", "proposed_status", "reviewer_id",
    "reviewer_role", "reviewer_action", "final_status", "review_notes",
    "notification_status", "error_details"
]

def append_audit(event):
    # Append-only during normal execution; previous audit rows are preserved.
    exists = AUDIT_PATH.exists()
    with AUDIT_PATH.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=AUDIT_FIELDS, extrasaction="ignore")
        if not exists or AUDIT_PATH.stat().st_size == 0:
            writer.writeheader()
        writer.writerow({key: event.get(key, "") for key in AUDIT_FIELDS})

DEMO_REVIEW_ACTION = "HOLD"  # Change to APPROVE, EDIT, or REQUEST_INFO for demonstration.
REVIEWER_ID = "HR-DEMO-001"
REVIEWER_ROLE = "HR_REVIEWER"
print("Default human-review action:", DEMO_REVIEW_ACTION)

Default human-review action: HOLD


## 9. Run the end-to-end workflow

The notebook computes scores and recommendations for valid records, creates drafts, simulates the selected human-review action, and appends audit events. No email is sent; notification status remains `NOT_SENT` because dispatch is intentionally not connected.

In [9]:
start_time = time.perf_counter()
results = []
audit_events_this_run = []

for row in validation_rows:
    append_audit({
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "candidate_id": row["candidate_id"],
        "event": "VALIDATION",
        "validation_status": row["validation_status"],
        "error_details": "" if row["validation_status"] == "VALID" else row["validation_reason"],
        "rule_version": rules["rule_version"],
        "notification_status": "NOT_SENT"
    })

for candidate in valid_records:
    score, breakdown = calculate_score(candidate, rules)
    proposed_status = route_candidate(score, rules)
    draft = generate_notification(candidate, proposed_status)

    reviewer_action = DEMO_REVIEW_ACTION
    try:
        final_status = apply_human_review(
            proposed_status, reviewer_action, REVIEWER_ID, REVIEWER_ROLE,
            notes="Demonstration review; replace with an actual documented HR decision."
        )
        review_error = ""
    except PermissionError as exc:
        final_status = "REQUIRES_REVIEW"
        review_error = str(exc)

    result = {
        **candidate,
        "score": score,
        **breakdown,
        "rule_version": rules["rule_version"],
        "proposed_status": proposed_status,
        "notification_subject": draft["subject"],
        "notification_message": draft["message"],
        "notification_status": "NOT_SENT",
        "reviewer_action": reviewer_action,
        "reviewer_id": REVIEWER_ID,
        "final_status": final_status
    }
    results.append(result)

    audit_event = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "candidate_id": candidate["candidate_id"],
        "event": "SCORING_AND_REVIEW",
        "validation_status": "VALID",
        "score": score,
        "score_breakdown": json.dumps(breakdown, sort_keys=True),
        "rule_version": rules["rule_version"],
        "proposed_status": proposed_status,
        "reviewer_id": REVIEWER_ID,
        "reviewer_role": REVIEWER_ROLE,
        "reviewer_action": reviewer_action,
        "final_status": final_status,
        "review_notes": "Demonstration review; replace with an actual documented HR decision.",
        "notification_status": "NOT_SENT",
        "error_details": review_error
    }
    append_audit(audit_event)

elapsed_seconds = time.perf_counter() - start_time
results_df = pd.DataFrame(results)

print("========== RECRUITMENT WORKFLOW ==========")
if len(results_df):
    sample = results_df.iloc[0]
    print("Candidate ID:", sample["candidate_id"])
    print("Validation: VALID")
    print("Score:", sample["score"], "/ 100")
    print("Score breakdown:", {
        "skills": sample["skills_contribution"],
        "experience": sample["experience_contribution"],
        "assessment": sample["assessment_contribution"]
    })
    print("Proposed Status:", sample["proposed_status"])
    print("Notification: DRAFT (not sent)")
    print("Human Review:", sample["final_status"])
print("------------------------------------------")
print("Total records loaded:", len(loaded_candidates))
print("Valid records:", len(valid_records))
print("Invalid/quarantined records:", len(rejected_records))
print("Candidates processed:", len(results_df))
print("Human reviews pending/held:", int((results_df["final_status"] == "ON_HOLD").sum()) if len(results_df) else 0)
print(f"Execution time: {elapsed_seconds:.4f} seconds")
print("Audit log:", AUDIT_PATH)

========== RECRUITMENT WORKFLOW ==========
Candidate ID: C101
Validation: VALID
Score: 82.0 / 100
Score breakdown: {'skills': np.float64(34.0), 'experience': np.float64(21.0), 'assessment': np.float64(27.0)}
Proposed Status: SHORTLIST_REVIEW
Notification: DRAFT (not sent)
Human Review: ON_HOLD
------------------------------------------
Total records loaded: 100
Valid records: 95
Invalid/quarantined records: 5
Candidates processed: 95
Human reviews pending/held: 95
Execution time: 0.0203 seconds
Audit log: /content/RecruitmentWorkflow/audit_log.csv


## 10. Review the results

The following tables show the criterion-wise score breakdown and the distribution of proposed workflow statuses. Scores are decision-support signals only; they do not establish candidate suitability or fairness.

In [10]:
display_cols = [
    "candidate_id", "candidate_name", "score",
    "skills_contribution", "experience_contribution", "assessment_contribution",
    "proposed_status", "final_status", "notification_status"
]
display(results_df[display_cols].head(20))

print("Proposed status counts:")
display(results_df["proposed_status"].value_counts().rename_axis("status").reset_index(name="candidate_count"))

print("Audit log preview:")
audit_df = pd.read_csv(AUDIT_PATH)
display(audit_df.tail(10))

,candidate_id,candidate_name,score,skills_contribution,experience_contribution,assessment_contribution,proposed_status,final_status,notification_status
0,C101,Aarav Shah,82.0,34.0,21.0,27.0,SHORTLIST_REVIEW,ON_HOLD,NOT_SENT
1,C102,Diya Mehta,74.5,28.0,24.0,22.5,MANUAL_REVIEW,ON_HOLD,NOT_SENT
2,C103,Kabir Patil,55.0,22.0,15.0,18.0,MANUAL_REVIEW,ON_HOLD,NOT_SENT
3,C104,Ira Desai,100.0,40.0,30.0,30.0,SHORTLIST_REVIEW,ON_HOLD,NOT_SENT
4,C105,Neel Joshi,0.0,0.0,0.0,0.0,NOT_SHORTLISTED_REVIEW,ON_HOLD,NOT_SENT
5,C006,Anaya Kulkarni,75.0,30.0,22.5,22.5,SHORTLIST_REVIEW,ON_HOLD,NOT_SENT
6,C007,Rohan Mehta,50.0,20.0,15.0,15.0,MANUAL_REVIEW,ON_HOLD,NOT_SENT
7,C008,Meera Joshi,75.0,30.0,22.5,22.5,SHORTLIST_REVIEW,ON_HOLD,NOT_SENT
8,C014,Ira Patil,81.7,24.4,27.9,29.4,SHORTLIST_REVIEW,ON_HOLD,NOT_SENT
9,C015,Neel Rao,83.3,35.6,21.0,26.7,SHORTLIST_REVIEW,ON_HOLD,NOT_SENT


Proposed status counts:


,status,candidate_count
0,MANUAL_REVIEW,64
1,SHORTLIST_REVIEW,22
2,NOT_SHORTLISTED_REVIEW,9


Audit log preview:


,timestamp,candidate_id,event,validation_status,score,score_breakdown,rule_version,proposed_status,reviewer_id,reviewer_role,reviewer_action,final_status,review_notes,notification_status,error_details
185,2026-10-21T06:46:45.336610+00:00,C091,SCORING_AND_REVIEW,VALID,61.3,"{""assessment_contribution"": 9.0, ""experience_c...",1.0,MANUAL_REVIEW,HR-DEMO-001,HR_REVIEWER,HOLD,ON_HOLD,Demonstration review; replace with an actual d...,NOT_SENT,NaN
186,2026-10-21T06:46:45.336716+00:00,C092,SCORING_AND_REVIEW,VALID,51.6,"{""assessment_contribution"": 14.4, ""experience_...",1.0,MANUAL_REVIEW,HR-DEMO-001,HR_REVIEWER,HOLD,ON_HOLD,Demonstration review; replace with an actual d...,NOT_SENT,NaN
187,2026-10-21T06:46:45.336820+00:00,C093,SCORING_AND_REVIEW,VALID,62.6,"{""assessment_contribution"": 11.1, ""experience_...",1.0,MANUAL_REVIEW,HR-DEMO-001,HR_REVIEWER,HOLD,ON_HOLD,Demonstration review; replace with an actual d...,NOT_SENT,NaN
188,2026-10-21T06:46:45.336956+00:00,C094,SCORING_AND_REVIEW,VALID,70.6,"{""assessment_contribution"": 12.9, ""experience_...",1.0,MANUAL_REVIEW,HR-DEMO-001,HR_REVIEWER,HOLD,ON_HOLD,Demonstration review; replace with an actual d...,NOT_SENT,NaN
189,2026-10-21T06:46:45.337066+00:00,C095,SCORING_AND_REVIEW,VALID,51.3,"{""assessment_contribution"": 16.2, ""experience_...",1.0,MANUAL_REVIEW,HR-DEMO-001,HR_REVIEWER,HOLD,ON_HOLD,Demonstration review; replace with an actual d...,NOT_SENT,NaN
190,2026-10-21T06:46:45.337184+00:00,C096,SCORING_AND_REVIEW,VALID,51.1,"{""assessment_contribution"": 12.6, ""experience_...",1.0,MANUAL_REVIEW,HR-DEMO-001,HR_REVIEWER,HOLD,ON_HOLD,Demonstration review; replace with an actual d...,NOT_SENT,NaN
191,2026-10-21T06:46:45.337291+00:00,C097,SCORING_AND_REVIEW,VALID,58.1,"{""assessment_contribution"": 11.4, ""experience_...",1.0,MANUAL_REVIEW,HR-DEMO-001,HR_REVIEWER,HOLD,ON_HOLD,Demonstration review; replace with an actual d...,NOT_SENT,NaN
192,2026-10-21T06:46:45.337401+00:00,C098,SCORING_AND_REVIEW,VALID,85.9,"{""assessment_contribution"": 28.2, ""experience_...",1.0,SHORTLIST_REVIEW,HR-DEMO-001,HR_REVIEWER,HOLD,ON_HOLD,Demonstration review; replace with an actual d...,NOT_SENT,NaN
193,2026-10-21T06:46:45.337508+00:00,C099,SCORING_AND_REVIEW,VALID,69.6,"{""assessment_contribution"": 26.1, ""experience_...",1.0,MANUAL_REVIEW,HR-DEMO-001,HR_REVIEWER,HOLD,ON_HOLD,Demonstration review; replace with an actual d...,NOT_SENT,NaN
194,2026-10-21T06:46:45.337619+00:00,C100,SCORING_AND_REVIEW,VALID,84.9,"{""assessment_contribution"": 18.3, ""experience_...",1.0,SHORTLIST_REVIEW,HR-DEMO-001,HR_REVIEWER,HOLD,ON_HOLD,Demonstration review; replace with an actual d...,NOT_SENT,NaN


## 11. Automated testing

Tests cover score calculations, routing boundaries, validation failures, duplicate detection, notification drafts, human-review actions, unauthorized approval, and missing-file handling.

In [11]:
test_results = []

def run_test(name, fn):
    try:
        fn()
        test_results.append({"test_case": name, "result": "PASS", "details": ""})
    except Exception as exc:
        test_results.append({"test_case": name, "result": "FAIL", "details": str(exc)})

def test_score_example():
    score, breakdown = calculate_score(
        {"skills_score":85, "experience_score":70, "assessment_score":90}, rules
    )
    assert score == 82.0
    assert breakdown["skills_contribution"] == 34.0
    assert breakdown["experience_contribution"] == 21.0
    assert breakdown["assessment_contribution"] == 27.0

def test_thresholds():
    assert route_candidate(75, rules) == "SHORTLIST_REVIEW"
    assert route_candidate(74.99, rules) == "MANUAL_REVIEW"
    assert route_candidate(50, rules) == "MANUAL_REVIEW"
    assert route_candidate(49.99, rules) == "NOT_SHORTLISTED_REVIEW"

def test_missing_email():
    seen = set()
    record = dict(candidates[8])
    _, ok, reason = validate_candidate(record, seen)
    assert not ok and ("Missing field: email" in reason)

def test_out_of_range():
    record = dict(candidates[9])
    _, ok, reason = validate_candidate(record, set())
    assert not ok and "between 0 and 100" in reason

def test_duplicate_detection():
    seen = set()
    first = dict(candidates[0])
    _, ok1, _ = validate_candidate(first, seen)
    duplicate = dict(candidates[12])
    _, ok2, reason2 = validate_candidate(duplicate, seen)
    assert ok1 and not ok2 and "Duplicate" in reason2

def test_notification_draft():
    candidate = {"candidate_id":"T001", "candidate_name":"Test Candidate"}
    draft = generate_notification(candidate, "NOT_SHORTLISTED_REVIEW")
    assert draft["notification_status"] == "DRAFT"
    assert "undergoing review" in draft["message"]
    assert "score" not in draft["message"].lower()

def test_review_actions():
    assert review_action("SHORTLIST_REVIEW", "APPROVE") == "SHORTLIST_REVIEW"
    assert review_action("SHORTLIST_REVIEW", "EDIT") == "REQUIRES_REVIEW"
    assert review_action("SHORTLIST_REVIEW", "HOLD") == "ON_HOLD"
    assert review_action("SHORTLIST_REVIEW", "REQUEST_INFO") == "AWAITING_INFORMATION"

def test_unauthorized_approval():
    try:
        apply_human_review("SHORTLIST_REVIEW", "APPROVE", "guest", "CANDIDATE")
    except PermissionError:
        return
    raise AssertionError("Unauthorized reviewer was allowed")

def test_missing_file():
    try:
        load_candidates(BASE_DIR / "does_not_exist.csv")
    except FileNotFoundError:
        return
    raise AssertionError("Missing file was not handled")

run_test("Weighted score and breakdown", test_score_example)
run_test("Routing threshold boundaries", test_thresholds)
run_test("Missing required email", test_missing_email)
run_test("Out-of-range score", test_out_of_range)
run_test("Duplicate candidate ID", test_duplicate_detection)
run_test("Notification draft safety", test_notification_draft)
run_test("Human review actions", test_review_actions)
run_test("Unauthorized approval denied", test_unauthorized_approval)
run_test("Missing candidate file handling", test_missing_file)

tests_df = pd.DataFrame(test_results)
display(tests_df)
tests_passed = int((tests_df["result"] == "PASS").sum())
total_tests = len(tests_df)
print(f"Test pass rate: {tests_passed}/{total_tests} = {tests_passed/total_tests*100:.2f}%")

,test_case,result,details
0,Weighted score and breakdown,PASS,
1,Routing threshold boundaries,PASS,
2,Missing required email,PASS,
3,Out-of-range score,PASS,
4,Duplicate candidate ID,PASS,
5,Notification draft safety,PASS,
6,Human review actions,PASS,
7,Unauthorized approval denied,PASS,
8,Missing candidate file handling,PASS,


Test pass rate: 9/9 = 100.00%


## 12. Evaluation metrics

Metrics follow the lab sheet. The validation rate is valid records divided by all ingested records; workflow processing success is successfully processed valid records divided by valid records. Notification draft generation is measured against valid records requiring a draft. No actual dispatch is performed, so delivery success/failure is not applicable.

In [12]:
total_records = len(loaded_candidates)
valid_count = len(valid_records)
invalid_count = len(rejected_records)
duplicate_count = sum("Duplicate candidate_id" in r.get("rejection_reason", "") for r in rejected_records)
processed_count = len(results_df)
drafts_generated = int(results_df["notification_message"].notna().sum()) if processed_count else 0

validation_rate = (valid_count / total_records * 100) if total_records else 0
workflow_success_rate = (processed_count / valid_count * 100) if valid_count else 0
draft_generation_rate = (drafts_generated / valid_count * 100) if valid_count else 0
test_pass_rate = (tests_passed / total_tests * 100) if total_tests else 0
avg_seconds_per_valid = (elapsed_seconds / valid_count) if valid_count else 0
reviewed_count = int((results_df["final_status"] != "ON_HOLD").sum()) if processed_count else 0
review_completion_rate = (reviewed_count / processed_count * 100) if processed_count else 0

status_counts = results_df["proposed_status"].value_counts().to_dict() if processed_count else {}
metrics = pd.DataFrame([
    ["Total candidate records", total_records],
    ["Valid records", valid_count],
    ["Invalid/quarantined records", invalid_count],
    ["Duplicate records detected", duplicate_count],
    ["Candidates in shortlist review", status_counts.get("SHORTLIST_REVIEW", 0)],
    ["Candidates in manual review", status_counts.get("MANUAL_REVIEW", 0)],
    ["Candidates in not-shortlisted review", status_counts.get("NOT_SHORTLISTED_REVIEW", 0)],
    ["Notification drafts generated", drafts_generated],
    ["Human approvals / edits / holds", processed_count],
    ["Notification failures", 0],
    ["Total execution time (seconds)", round(elapsed_seconds, 4)],
    ["Test cases passed", f"{tests_passed}/{total_tests}"],
    ["Data validation rate (%)", round(validation_rate, 2)],
    ["Workflow processing success rate (%)", round(workflow_success_rate, 2)],
    ["Notification draft generation rate (%)", round(draft_generation_rate, 2)],
    ["Test pass rate (%)", round(test_pass_rate, 2)],
    ["Average processing time per valid record (seconds)", round(avg_seconds_per_valid, 6)],
    ["Human review completion rate (%)", round(review_completion_rate, 2)]
], columns=["Metric", "Observed value"])
display(metrics)

,Metric,Observed value
0,Total candidate records,100
1,Valid records,95
2,Invalid/quarantined records,5
3,Duplicate records detected,1
4,Candidates in shortlist review,22
5,Candidates in manual review,64
6,Candidates in not-shortlisted review,9
7,Notification drafts generated,95
8,Human approvals / edits / holds,95
9,Notification failures,0


## 13. Security, fairness, and reliability notes

- **Privacy:** use only synthetic candidate records in this lab; avoid unnecessary personal information.
- **Access control:** only `HR_REVIEWER` and `HR_ADMIN` roles are authorized by the demonstration function. A production system needs real authentication and authorization.
- **Prompt/data injection:** candidate-provided text is treated as data, not as instructions. It cannot modify the external `rules.json` configuration.
- **Human oversight:** scores and routing labels are recommendations; reviewers must document overrides and make final decisions.
- **Auditability:** audit events are appended to `audit_log.csv`, while invalid records and reasons are stored in `rejected_records.csv`.
- **Idempotency limitation:** this lab appends audit events on each run. A production deployment should use a stable import/run ID and database uniqueness constraints to avoid accidental repeated processing.
- **Notification limitation:** messages remain drafts and no external email service is connected.
- **Fairness limitation:** passing software tests does not prove the criteria or outcomes are fair. The illustrative weights and thresholds require role-specific validation and ongoing review.

## 14. Download generated outputs

Files are stored in `/content/RecruitmentWorkflow/` in Colab. The cell below bundles the generated CSV/JSON outputs into a ZIP for convenient download.

In [13]:
import zipfile
from IPython.display import FileLink, display

OUTPUT_FILES = [
    BASE_DIR / "candidates.csv",
    BASE_DIR / "rules.json",
    BASE_DIR / "rejected_records.csv",
    BASE_DIR / "audit_log.csv"
]
ZIP_PATH = Path("/content/RecruitmentWorkflow_outputs.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_FILES:
        if file_path.exists():
            zf.write(file_path, arcname=file_path.name)

print("Output archive created:", ZIP_PATH)
display(FileLink(str(ZIP_PATH)))

Output archive created: /content/RecruitmentWorkflow_outputs.zip


/content/RecruitmentWorkflow_outputs.zip

## Conclusion

The notebook implements an automated recruitment workflow with CSV ingestion, normalization and validation, duplicate detection, configurable weighted scoring, criterion-wise explanations, routing, notification drafts, human review controls, audit logging, automated tests, and performance metrics. It demonstrates how repetitive administrative tasks can be automated while retaining transparency, traceability, privacy, and human oversight.